# Hugging Face GPU Eval Notebook

Use this notebook to evaluate the APUSH LoRA directly from Hugging Face on a notebook GPU. It compares three generators: Qwen base, APUSH-tuned LoRA, and the teacher API model. The grading judge also uses the API token when enabled.

Recommended order:
1. Set `BASE_MODEL_ID`, `APUSH_ADAPTER_ID`, and API-token env vars.
2. Install/load the HF inference dependencies.
3. Run the tokenizer/template smoke test.
4. Run the one-prompt generation smoke test.
5. Run the smoke eval across all three generators.
6. Flip `RUN_FULL_EVAL = True` only when the smoke looks clean.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import textwrap
import time

try:
    import pandas as pd
except Exception:
    pd = None

def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "eval" / "harness.py").exists() and (p / "prompts" / "litmus_generation_prompt.md").exists():
            return p
    raise RuntimeError("Could not find repo root. Run this notebook from inside the slm repo.")

ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "eval"))
print("repo:", ROOT)
print("python:", sys.executable)

## Configure Models

`PROMPTLENS_API_KEY` is used for the teacher generator and the judge. `HF_TOKEN` is only needed if the base or adapter repo is private/gated. With `USE_JUDGE = False`, the notebook still checks formatting and programmatic gates, but the verdict is not a real expert-quality eval.

In [ ]:
BASE_MODEL_ID = "unsloth/Qwen3-4B"       # exact base used in train/qlora_qwen3_4b.ipynb
APUSH_ADAPTER_ID = "rohanpalviela/qwen3-4b-apush-v2-clean-lora"

INSTALL_HF_DEPS = True
LOAD_IN_4BIT = True                      # recommended for Colab T4/L4 VRAM
USE_API = bool(os.environ.get("PROMPTLENS_API_KEY"))
USE_JUDGE = USE_API
RUN_TEACHER = USE_API                    # include teacher as the third generator when API key is available

JUDGE_CFG = {
    "name": "gpt-5.5",
    "provider": "openai_compatible",
    "model": "openai-group/gpt-5.5",
    "base_url": "https://tfy.promptlens.trilogy.com/v1",
    "api_key_env": "PROMPTLENS_API_KEY",
    "max_tokens": 2048,
    "include_temperature": False,
    "token_param": "max_completion_tokens",
}

TEACHER_CFG = {
    "name": "claude-opus-4-8",
    "provider": "openai_compatible",
    "model": "claude-group/claude-opus-4-8",
    "base_url": "https://tfy.promptlens.trilogy.com/v1",
    "api_key_env": "PROMPTLENS_API_KEY",
    "max_tokens": 8192,
}

def hf_candidate(name, *, model=BASE_MODEL_ID, adapter=None):
    return {
        "name": name,
        "provider": "hf_local",
        "model": model,
        "adapter": adapter,
        "think": False,
        "load_in_4bit": LOAD_IN_4BIT,
        "torch_dtype": "float16",
        "device_map": "auto",
        "hf_token_env": "HF_TOKEN",
        "max_tokens": 1536,
        "temperature": 0.7,
    }

candidates = [
    hf_candidate("qwen3-4b-base"),
    hf_candidate("qwen3-apush-tuned", adapter=APUSH_ADAPTER_ID),
]

models = {"candidates": candidates}
if RUN_TEACHER:
    models["teacher"] = TEACHER_CFG
if USE_JUDGE:
    models["judge"] = JUDGE_CFG

CONFIG_PATH = ROOT / "eval" / "models.notebook.json"
CONFIG_PATH.write_text(json.dumps(models, indent=2) + "\n", encoding="utf-8")
print(CONFIG_PATH)
print(json.dumps(models, indent=2))
if not USE_API:
    print("\nNo PROMPTLENS_API_KEY found: notebook will run base+tuned only and add --no-judge by default.")

In [ ]:
def run_cmd(cmd, *, check=True, timeout=None):
    print("+", " ".join(str(x) for x in cmd))
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    if proc.stdout:
        print(proc.stdout.rstrip())
    if check and proc.returncode:
        raise RuntimeError(f"command failed with exit {proc.returncode}: {cmd}")
    return proc

if INSTALL_HF_DEPS:
    run_cmd([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers", "accelerate", "peft", "bitsandbytes", "sentencepiece", "huggingface_hub",
    ], timeout=None)

try:
    import torch
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch check failed:", e)

## Check Hugging Face Access

This verifies the base model and APUSH adapter repos are visible before the first generation call downloads model weights. If either repo is private/gated, set `HF_TOKEN` in the environment first.

The actual model weights are downloaded lazily when `hf_local` generates for the first time.

In [ ]:
from huggingface_hub import model_info

for repo_id in [BASE_MODEL_ID, APUSH_ADAPTER_ID]:
    info = model_info(repo_id, token=os.environ.get("HF_TOKEN") or None)
    siblings = [s.rfilename for s in info.siblings]
    interesting = [s for s in siblings if s.endswith(("adapter_config.json", "adapter_model.safetensors", "config.json", "tokenizer_config.json"))]
    print(repo_id)
    print("  files:", interesting[:10])

## Inspect the Qwen Chat Template

This checks the tokenizer chat template path used by the HF local provider. For Qwen3, the provider passes `enable_thinking=False` when the tokenizer supports it.

In [ ]:
from transformers import AutoTokenizer

try:
    tokenizer = AutoTokenizer.from_pretrained(APUSH_ADAPTER_ID, token=os.environ.get("HF_TOKEN") or None, trust_remote_code=True)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, token=os.environ.get("HF_TOKEN") or None, trust_remote_code=True)
messages = [
    {"role": "system", "content": "Return only JSON."},
    {"role": "user", "content": "Say OK as a JSON array."},
]
try:
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
except TypeError:
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(rendered[:2000])
print("\n--- quick flags ---")
checks = {
    "contains_<think>": "<think" in rendered.lower(),
    "contains_/think_suffix": "/think" in rendered.lower(),
    "contains_no_think_suffix": "/no_think" in rendered.lower(),
    "has_im_start": "<|im_start|>" in rendered,
    "has_tool_scaffold": "<tools>" in rendered or "tool_call" in rendered,
}
for k, v in checks.items():
    print(f"{k:28} {v}")

if checks["contains_<think>"] or checks["contains_/think_suffix"]:
    print("\nWARNING: rendered prompt appears to mention thinking. Inspect before trusting eval numbers.")

## HF Local Generation Smoke Test

This uses the exact repo prompt builder plus the `hf_local` provider. It loads the Qwen base model with the APUSH adapter on GPU, disables Qwen3 thinking through the tokenizer template when available, prints the raw model response, and runs the programmatic gates on parsed items.

In [ ]:
import providers
import checks as programmatic_checks
from prompt_loader import LitmusPrompt, extract_items, canonicalize_item_archetype
from harness import load_sources, DIFFICULTY

prompt = LitmusPrompt.from_file(ROOT / "prompts" / "litmus_generation_prompt.md")
source = load_sources("LITMUS")[0]
requested_archetype = "CAUSE_OF_SOURCE"
system, user = prompt.build(
    source=source["text"],
    attribution=source.get("attribution", ""),
    note="",
    n=1,
    archetypes=requested_archetype,
    difficulty=DIFFICULTY,
    include_fewshot=False,
)

cfg = hf_candidate("qwen3-apush-tuned", adapter=APUSH_ADAPTER_ID)
raw = providers.generate(cfg, system, user, temperature=0.0, role="candidate")
print("--- raw response ---")
print(raw)

print("\n--- format flags ---")
stripped = raw.strip()
format_flags = {
    "starts_with_json_array": stripped.startswith("["),
    "contains_think_tag": "<think" in stripped.lower() or "</think" in stripped.lower(),
    "contains_markdown_fence": "```" in stripped,
    "contains_preface_text_before_json": bool(stripped) and not stripped.startswith(("[", "{")),
}
for k, v in format_flags.items():
    print(f"{k:32} {v}")

items = [canonicalize_item_archetype(it, requested_archetype=requested_archetype) for it in extract_items(raw)]
print(f"\nparsed items: {len(items)}")
for i, item in enumerate(items, start=1):
    prog = programmatic_checks.run_checks(item, source)
    print(f"\nitem {i}")
    print(json.dumps(item, indent=2)[:2500])
    print("programmatic:", json.dumps(prog, indent=2))

## Harness Dry Run

This proves the eval plumbing without touching Hugging Face model weights or external APIs. The numbers are fake.

In [ ]:
run_cmd([sys.executable, "eval/harness.py", "--dry-run", "--runs", "1", "--limit", "1", "--n", "2", "--out", "results/notebook_dry_run"], timeout=180)

## Smoke Eval

Small, cheap run over two sources. If `USE_JUDGE` is false, this runs programmatic checks only.

In [ ]:
smoke_cmd = [
    sys.executable,
    "eval/harness.py",
    "--models", str(CONFIG_PATH),
    "--runs", "1",
    "--limit", "2",
    "--n", "2",
    "--out", "results/notebook_smoke",
]
if not USE_JUDGE:
    smoke_cmd.append("--no-judge")
run_cmd(smoke_cmd, timeout=1800)

## Full Eval

Flip `RUN_FULL_EVAL` after the smoke test is clean. Local HF GPU generation runs serially to avoid VRAM contention, so this can still take a while.

In [ ]:
RUN_FULL_EVAL = False

if RUN_FULL_EVAL:
    full_cmd = [
        sys.executable,
        "eval/harness.py",
        "--models", str(CONFIG_PATH),
        "--runs", "3",
        "--n", "6",
        "--out", "results/notebook_full",
    ]
    if not USE_JUDGE:
        full_cmd.append("--no-judge")
    run_cmd(full_cmd, timeout=None)
else:
    print("RUN_FULL_EVAL is False. Change it to True after the smoke eval passes.")

## Read Results

In [ ]:
RESULT_DIR = ROOT / "results" / ("notebook_full" if RUN_FULL_EVAL else "notebook_smoke")
report_path = RESULT_DIR / "litmus_results.md"
items_path = RESULT_DIR / "litmus_items.jsonl"

if report_path.exists():
    print(report_path)
    print(report_path.read_text(encoding="utf-8")[:6000])
else:
    print("No report yet:", report_path)

In [ ]:
if items_path.exists():
    rows = [json.loads(line) for line in items_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    print("items:", len(rows))
    cols = ["model", "source_id", "archetype", "near_miss", "key_valid", "answer", "stem"]
    if pd is not None:
        display(pd.DataFrame(rows)[[c for c in cols if c in rows[0]]])
    elif rows:
        print(json.dumps(rows[0], indent=2)[:4000])
else:
    print("No per-item file yet:", items_path)